# Benchmark Quantum Adder

**Quantum arithmetic** is a central ingredient in many quantum algorithms. Quantum circuits that manipulate encoded integers or numerical variables coherently rely on reversible arithmetic primitives such as addition, comparison, and multiplication. These building blocks appear prominently in Shor’s algorithm, where modular arithmetic is essential, and also in oracle-based settings related to Grover-style search, where arithmetic can be part of the problem encoding. In practice, arithmetic subroutines often contribute significantly to circuit depth and overall resource cost.

Among these primitives, the **quantum adder** is one of the simplest and most canonical examples. It implements the reversible transformation of one register by adding the value of another, making it a natural starting point for studying quantum arithmetic. It is also a basic component in more advanced arithmetic circuits, including multipliers, comparators, accumulators, and modular routines.

We benchmark an out-of-place addition by a constant:
$$
|x\rangle_n|0\rangle_n \rightarrow |x\rangle_n|x+c\, (\bmod 2^n) \rangle_n,
$$
where $c$ is a given integer.

***

### The model
We start with a uniform superposition for $|x\rangle$, and add the constant $c=2^{\lfloor n/2 \rfloor}-1$:
$$
\frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle_n|0\rangle_n \rightarrow \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle_n|x+c \, (\bmod 2^n)\rangle_n.
$$

### The Scoring function
We measure the probabilty of getting a correct result:
$$
{\rm Score = }\sum_{y,x=0}^{2^n-1} P\left(y= x+c \, (\bmod 2^n)\right),
$$
where $y$ is the measured value of the second variable.

***
***

In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *

In [2]:
# ============================================================
# Fresh start: reset all generated report/results files
# Uncomment to start a new benchmarking project from scratch
#
# from project_reset import reset_benchmark_project

# reset_benchmark_project()
# ============================================================

In [3]:
from examples.adder import AdderExample

***
***
## Benchmarking a 4-qubits adder

Define a specific example on 4 qubits (note that the models are optimized for width):

In [4]:
example = AdderExample(problem_size=4)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3DLMuGDT13HPEHGztznfOJmQD40


### Set a `ResultCollector` with a file name fixed for the current example

In [5]:
FILENAME = example.default_results_filename

In [6]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [7]:
# Uncomment to clear the data of a previous run
#
# await collector.reset_file()

### Choose on which backend to run 

We can print the list of backends:

In [8]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [9]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [10]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [11]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [12]:
print(
    "=" * 10
    + f"  {example.name}-{example.problem_size} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  adder-4 (2026-05-06 11:45:18.941596)   ==========
2026-05-06 11:45:22.960730: Submit adder-4 for Classiq - simulator
2026-05-06 11:45:25.923260: Completed adder-4 for Classiq - simulator --> Score {'score': '1.000', 'execution_time': '0.013', 'circuit_depth': 102, 'circuit_width': 9, 'two_qubit_gate_count': 84}
2026-05-06 11:45:27.180835: Submit adder-4 for Classiq - simulator_statevector
** Report updated: adder-4 for Classiq - simulator
2026-05-06 11:45:29.669607: Completed adder-4 for Classiq - simulator_statevector --> Score {'score': '1.000', 'execution_time': '0.015', 'circuit_depth': 103, 'circuit_width': 9, 'two_qubit_gate_count': 84}
** Report updated: adder-4 for Classiq - simulator_statevector


In [13]:
await collector.print_status()

========== (2026-05-06 11:45:30.516239)   ==========
adder-4 | Classiq - simulator | COMPLETED | score=1.000 | time=0.013 min
adder-4 | Classiq - simulator_statevector | COMPLETED | score=1.000 | time=0.015 min
